In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")          # reads ANTHROPIC_API_KEY from .env into the environment
client = anthropic.Anthropic()  # finds the key automatically; it never appears in your code

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(response.content[0].text)

Hello there, friend. How are you?


In [5]:
SYSTEM = """You answer questions about board and card game rules, using ONLY the rulebook excerpts provided.

Rules:
1. Use only the excerpts. Do not use outside knowledge, even if you know the game well.
2. Each excerpt is labeled with its game. Only use excerpts from the game the question asks about.
   If the question names no game, answer separately for each game the excerpts cover, labeled by game.
   If the question uses a term that belongs to only one game (e.g. "hole cards" for Texas Hold'em), treat it as about that game.
3. After each claim, cite the excerpt id in square brackets, e.g. [catan-p9-c0].
4. If the excerpts do not contain the answer, reply exactly: "The provided rules don't cover this."
   You may then mention the closest related rule, clearly labeled as related.
5. Be short and direct."""

In [7]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":   # only move up if we're still in notebooks/
    os.chdir("..")
sys.path.append("src")
print("Working in:", os.getcwd())    # should end in /Game
from rulebook_rag.retrieve import Retriever

retriever = Retriever()

def format_context(chunks):
    parts = []
    for c in chunks:
        parts.append(f"<excerpt id=\"{c['id']}\" game=\"{c['game']}\" page=\"{c['page']}\">\n{c['text']}\n</excerpt>")
    return "\n\n".join(parts)

def answer(question, k=3):
    chunks = retriever.search(question, k=k)
    user_message = f"Excerpts:\n{format_context(chunks)}\n\nQuestion: {question}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    text = "".join(b.text for b in response.content if b.type == "text")
    return text, chunks

Working in: /Users/yasasvikotagiri/Desktop/Game


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10824.94it/s]


In [8]:
for q in [
    "How many private hole cards is each player dealt?",           # direct lookup
    "Do you collect money for landing on Free Parking?",           # myth-buster
    "What penalty does the CATAN rulebook impose for flipping the board?",  # unanswerable
]:
    text, chunks = answer(q)
    print("Q:", q)
    print("A:", text)
    print("Chunks:", [c["id"] for c in chunks])
    print("=" * 60)

Q: How many private hole cards is each player dealt?
A: Each player is dealt two private hole cards [texas_holdem-p2-c1]. The deal also confirms that every player receives two cards for their eyes only [texas_holdem-p2-c0].
Chunks: ['texas_holdem-p2-c1', 'rummy-p4-c1', 'texas_holdem-p2-c0']
Q: Do you collect money for landing on Free Parking?
A: No. The rules say you don't get a bonus for landing on Free Parking [monopoly-p1-c0]. The Free Parking space reads "Relax! Nothing happens." [monopoly-p2-c5]
Chunks: ['monopoly-p1-c0', 'monopoly-p2-c5', 'monopoly-p2-c3']
Q: What penalty does the CATAN rulebook impose for flipping the board?
A: The provided rules don't cover this.

Related: the Catan excerpts describe only the variable setup (placing hexes, number discs, and the robber) and the turn structure (Production phase, then Action phase) [catan-p11-c0] [catan-p6-c0]. They say nothing about penalties for flipping the board.
Chunks: ['catan-p11-c0', 'catan-p6-c0', 'texas_holdem-p3-c2']


In [9]:
import csv

with open("eval/questions.csv") as f:
    questions = list(csv.DictReader(f))

results = []
for row in questions:
    text, chunks = answer(row["question"])
    results.append({
        "id": row["id"],
        "question_type": row["question_type"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "answer": text,
        "chunks": ";".join(c["id"] for c in chunks),
    })
    print(row["id"], "done")

with open("eval/results_v2.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print("Saved", len(results), "results")

1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
10 done
11 done
12 done
13 done
14 done
15 done
16 done
17 done
18 done
19 done
20 done
21 done
22 done
23 done
24 done
25 done
26 done
27 done
28 done
29 done
30 done
31 done
32 done
33 done
34 done
35 done
36 done
37 done
38 done
39 done
40 done
41 done
Saved 41 results


In [10]:
for r in results:
    if r["question_type"] in ("unanswerable", "myth_buster"):
        print(f"[{r['id']}] {r['question_type']}")
        print("Q:       ", r["question"])
        print("EXPECTED:", r["expected_answer"])
        print("GOT:     ", r["answer"])
        print("-" * 60)

[3] myth_buster
Q:        Is every development card forbidden from being used on the turn you buy it?
EXPECTED: No. Victory Point cards bought that turn may be revealed if they give you enough points to win.
GOT:      No. Most development cards can't be played on the turn you buy them, but Victory Point cards are an exception when you're trying to win.

- Most development cards: "It may not be a card you built this turn." [catan-p9-c0]
- Victory Point cards: "You may play multiple VP cards (even on the turn you buy them) in order to win the game." [catan-p3-c1]
------------------------------------------------------------
[5] unanswerable
Q:        What penalty does the CATAN rulebook impose for flipping over the board?
EXPECTED: Not specified in the supplied rulebook.
GOT:      The provided rules don't cover this.

Related (not a penalty): The setup rules say to place the hexes face up and the number discs face up after turning them over [catan-p11-c0], and the turn overview describes 

In [11]:
by_id = {c["id"]: c for c in retriever.chunks}
print(by_id["texas_holdem-p2-c2"]["text"])

In Hold'em, a marker called ‘the button' or ‘the dealer button' indicates which player is the dealer for the current game. Before the game begins, the player immediately clockwise from the button posts the "small blind", the first forced bet. The player immediately clockwise from the small blind posts the "big blind", which is typically twice the size of the small blind, but the blinds can vary depending on the stakes and betting structure being played.

In Limit games, the big blind is the same as the small bet, and the small blind is typically half the size of the big blind but may be larger depending on the stakes. For example, in a 4 Limit 2/ game the small blind is $1 and the big blind is $2. In a 30 Limit game, the small blind is $10 15/ and the big blind is $15.


In [12]:
print("1:1" in by_id["blackjack-p2-c0"]["text"])

True


## Faithfulness v2

- Unanswerable: 6/6 correct refusals
- Myth-buster: 6/6 correct, all cited
- Related-rule tails: checked rows 22 and 36 against the chunks — grounded
- New extraction bug: Hold'em "$15/$30 Limit" garbled to "30 Limit ... $10 15/"